# 강의 06 · 실습 11 — 패턴 5 오케스트레이터-워커 · (3) 변형

## 1. 문제상황

- 고객지원팀은 신제품이 나올 때마다 제품 자료를 읽고 자주 묻는 질문(FAQ) 초안을 만듭니다.
- 질문의 수와 내용은 제품 자료에 따라 다릅니다. 기능이 단순한 제품은 질문 두 개면 되고, 기능이 많은 제품은 질문 네 개가 필요합니다.
- 담당자는 자료를 읽고 질문을 먼저 뽑은 뒤, 질문마다 자료를 다시 보며 답을 쓰고, 질문 번호 순서대로 정리합니다.
- 질문을 뽑는 일과 답을 쓰는 일이 제품마다 반복되고, 답을 여러 사람이 나눠 쓰면 순서가 뒤섞입니다.

## 2. 문제와 목표

- **문제**: 하위 작업(질문)의 개수가 입력(제품 자료)에 따라 달라지고, 답을 나눠 쓰면 완성본의 순서가 보장되지 않습니다.
- **목표**: 제품 자료 원문을 입력하면 orchestrator가 번호 붙은 질문 계획을 구조화 출력으로 세우고, 질문 수만큼 worker를 팬아웃해 각 worker가 자료 원문과 자기 질문을 받아 답을 쓰게 한 뒤, synthesizer가 번호순으로 정렬해 FAQ로 이어 붙이는 처리 흐름을 만듭니다.
    - orchestrator: 자료 원문을 읽고 번호 붙은 질문 2~4개를 구조화 출력 `Questions`로 계획합니다.
    - worker: 질문 하나와 자료 원문을 받아 자료 안의 내용만으로 두 문장 답을 씁니다(`Send`로 팬아웃).
    - synthesizer: 모델을 부르지 않고 부분 결과를 번호순으로 정렬한 뒤 「Q번호. 질문 / A. 답」으로 이어 붙입니다.
- **목표 달성 여부의 판정 기준**: 제품 자료를 입력했을 때, 계획한 질문의 수만큼 답이 따로 만들어지고, 완성 FAQ의 질문 번호가 1부터 순서대로 이어지는 것을 실행 결과에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec06_ex11_s3_diagram.svg)

## 4. 단계별 요구사항

1. **상태와 계획 규격을 정의합니다.**
    - 계획 규격 `Question`은 번호(`no`)와 질문 문장(`text`)을 가지고, `Questions`는 `Question`의 목록을 가집니다.
    - 부모 상태 `FaqState`는 자료 원문(`source`), 질문 계획(`questions`), 답한 부분 결과(`answered`), 완성 FAQ(`faq`) 키 네 개를 가지며, `answered`에는 `operator.add` 리듀서를 겁니다.
    - worker 전용 상태 `WorkerState`는 자기 질문(`question`), 자료 원문(`source`), 같은 리듀서를 건 `answered` 키 세 개를 가집니다.
2. **orchestrator 노드를 만듭니다.**
    - `Questions` 규격을 건 모델을 불러 「제품 자료를 읽고 고객이 자주 물을 질문을 필요한 만큼(2~4개) 번호를 붙여 계획한다」는 지침으로 계획을 받고, 그 목록을 `questions` 키에 씁니다.
3. **worker 노드를 만듭니다.**
    - worker는 `WorkerState`를 받아 자료 원문과 자기 질문을 모델에 넣고 「자료에 있는 내용만으로 질문에 두 문장으로 답한다」는 지침으로 답을 쓴 뒤, 번호·질문·답을 담은 딕셔너리 하나를 리스트에 담아 `answered` 키로 돌려줍니다.
4. **synthesizer 노드를 만듭니다.**
    - synthesizer는 모델을 부르지 않고, `answered` 부분 결과를 번호순으로 정렬한 뒤 「Q1. 질문 / A. 답」 형식으로 이어 붙여 `faq` 키에 씁니다.
5. **그래프에 노드를 등록합니다.**
    - orchestrator·worker·synthesizer 세 노드를 이름과 함께 등록합니다.
6. **엣지를 연결합니다.**
    - START에서 orchestrator로 가는 고정 엣지를 추가합니다.
    - orchestrator 뒤에는 판단 함수 assign_workers가 질문마다 `Send("worker", {"question": 항목, "source": 자료 원문})`을 돌려주는 조건부 엣지를 추가합니다.
    - worker 뒤에는 synthesizer를, synthesizer 뒤에는 END를 고정 엣지로 연결합니다.
7. **그래프를 컴파일하고 실행합니다.**
    - 제품 자료 원문과 빈 계획, 빈 부분 결과 목록, 빈 FAQ를 넣어 실행한 뒤, 팬아웃된 worker의 수와 완성 FAQ를 출력합니다.
    - 값(`SOURCE`, 제품 자료 원문)은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
    - orchestrator는 진입할 때 「[orchestrator] 진입 -> 질문 N개 계획」 줄을, worker는 「[worker] 진입: Q번호 질문」 줄을, synthesizer는 「[synthesizer] 진입 -> 부분 결과 N개를 번호순으로 정렬해 합침」 줄을 출력합니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 부모 상태·worker 전용 상태·계획 규격을 선언하고 리듀서를 겁니다 | `class FaqState(TypedDict)`, `Annotated[list, operator.add]` | 1 |
| ② 노드 함수 정의 | 질문을 계획하는 orchestrator, 답 하나를 쓰는 worker, 정렬해 합치는 synthesizer를 만듭니다 | `llm.with_structured_output(Questions)`, `def worker(state: WorkerState) -> dict` | 2, 3, 4 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 세 노드를 등록합니다 | `StateGraph(FaqState)`, `add_node` | 5 |
| ④ 엣지 연결 | 실행 시점에 worker를 팬아웃하는 Send 목록과 고정 순서를 정합니다 | `add_conditional_edges`, `Send("worker", {...})`, `add_edge` | 6 |
| ⑤ 컴파일과 실행 | 연결을 확정하고 자료 원문을 넣어 실행합니다 | `compile()`, `invoke()` | 7 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import operator
import os

from dotenv import load_dotenv, find_dotenv
from typing import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from pydantic import BaseModel, Field   # Field — 규격 필드에 설명(description=…)을 달 때 씁니다

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")

# 주어진 자료: 제품 자료 원문 SOURCE — 값을 그대로 씁니다
SOURCE = ("무선 이어폰 에어핏 프로. 한 번 충전으로 최대 8시간 재생, 충전 케이스 포함 32시간. "
          "IPX5 생활 방수. 블루투스 5.3, 기기 두 대 동시 연결. "
          "노이즈 캔슬링 3단계. 무상 보증 1년, 소모품인 이어팁은 보증 제외.")

### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 노드들이 읽고 쓰는 키를 선언합니다. 여러 worker가 같은 `answered` 키에 동시에 쓰므로 `operator.add` 리듀서가 필요합니다. `WorkerState`에는 자료 원문 `source` 키가 있습니다. worker가 답을 쓰려면 자기 질문만이 아니라 자료도 봐야 하기 때문입니다. 계획 규격 `Question`의 번호 `no`가 뒤에서 정렬의 기준이 됩니다.

In [ ]:
# 여기에 단계 ①(계획 규격 Question·Questions, 부모 상태 FaqState, worker 상태 WorkerState 정의)을 작성합니다.

### 단계 ② — 노드 함수 정의 (요구사항 2, 3, 4)

- orchestrator는 자료 원문을 읽고 번호 붙은 질문 목록을 구조화 출력으로 받습니다. 목록의 길이가 곧 worker의 수입니다.
- worker는 자기 질문과 자료 원문을 함께 받아 답을 쓰고, 번호·질문·답을 담은 딕셔너리 하나를 리스트 항목으로 돌려줍니다. worker들은 동시에 돌므로 부분 결과가 쌓이는 순서는 보장되지 않습니다.
- synthesizer는 모델을 부르지 않습니다. 부분 결과를 번호순으로 정렬한 뒤 이어 붙입니다. 순서를 보장하는 노드는 worker가 아니라 synthesizer입니다.

In [ ]:
# 여기에 단계 ②(orchestrator, worker, synthesizer 노드 정의)를 작성합니다.

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 5)

`StateGraph`에 부모 상태를 넘겨 빈 그래프를 열고, `add_node`로 함수마다 이름을 붙여 등록합니다. worker 노드는 한 번만 등록합니다. 몇 개로 팬아웃될지는 등록이 아니라 실행 시점의 `Send` 목록이 정합니다.

In [ ]:
# 여기에 단계 ③(그래프 빌더 생성과 노드 등록)을 작성합니다.

### 단계 ④ — 엣지 연결 (요구사항 6)

`add_edge`는 고정된 순서로 연결합니다. `add_conditional_edges`의 판단 함수가 노드 이름 대신 `Send` 목록을 돌려주면, 목록의 항목 수만큼 그 노드가 팬아웃됩니다. `Send` 한 쌍은 워커 이름과 그 워커가 받을 상태입니다. 세 번째 인자는 이 분기가 닿을 수 있는 노드를 그래프에 알려 주는 목록입니다. worker 뒤에는 synthesizer를, synthesizer 뒤에는 END를 고정 엣지로 연결합니다. `Send`에 담는 상태에 질문과 함께 자료 원문도 넣습니다.

In [ ]:
# 여기에 단계 ④(Send 목록을 돌려주는 판단 함수와 엣지 연결)를 작성합니다.

### 단계 ⑤ — 컴파일과 실행 (요구사항 7)

`compile()`이 연결을 확정해 실행 가능한 그래프를 돌려줍니다. `invoke`에 자료 원문과 빈 값들을 넣으면 최종 상태가 돌아옵니다. 아래에서는 무선 이어폰 제품 자료를 넣습니다.

In [ ]:
# 여기에 단계 ⑤(컴파일과 실행)를 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. `[orchestrator] 진입` 줄에 출력된 계획 질문 수와 `[worker] 진입` 줄의 수가 같습니다.
2. `[synthesizer] 진입` 줄이 worker 줄들 뒤에 한 번만 출력되고, 정렬해 합친 부분 결과 수가 worker 수와 같습니다.
3. 완성 FAQ의 질문 번호가 Q1부터 순서대로 이어집니다.

세 가지가 모두 확인되면 완성입니다. 하나라도 다르면 해당 단계의 코드를 다시 봅니다. FAQ의 번호가 뒤섞이면 단계 ②의 synthesizer 정렬을, worker가 자료 없이 답하면 단계 ④의 `Send`에 자료 원문을 넣었는지 다시 봅니다.